# Sistema de Teleoperación 3D Multimodo con Realimentación Táctil para Mano Robótica Inspire RH56DFTP-2R
Sistema de teleoperación tridimensional controlado por **Intel RealSense Depth Camera** para la mano robótica **Inspire RH56DFTP-2R** de 6 grados de libertad (DOF). En este documento se describe de forma exhaustiva, bloque por bloque, la arquitectura, el modelo cinemático 3D, las máquinas de estado de control por detección de presión y la comunicación mediante registros **Modbus TCP** empleando **MediaPipe Hands**.

**Nota:**
- Las dependencias requeridas por el entorno se encuentran especificadas en el archivo `requirements_telecontrol.txt` del repositorio.

---


## Bloque 1: Importación de Librerías, Concurrencia y Modo de Respaldo (Simulación)

1. **Ecosistema de Visión, Cálculo y Concurrencia**:
   - `pyrealsense2 (rs)`: SDK oficial de Intel RealSense encargado de capturar los flujos de color (RGB) y mapa de profundidad (`Z16`), alinearlos espacialmente y deproyectar píxeles 2D a coordenadas métricas 3D $[X, Y, Z]$.
   - `numpy (np)`: Motor de álgebra lineal empleado para el cálculo vectorizado de distancias euclidianas, diferenciales de profundidad, promedios estadísticos en calibración y normalización de consignas.
   - `cv2 (OpenCV)`: Renderizado de la interfaz gráfica de usuario (HUD), trazado del esqueleto (*wireframe*) y captura de eventos de teclado en tiempo real.
   - `mediapipe (mp)`: Red neuronal de inferencia ligera que detecta los 21 nodos articulares (*landmarks*) sobre la imagen RGB de la mano.
   - `threading`: Módulo nativo de concurrencia utilizado para desacoplar el bucle de visión ($30\,\text{FPS}$) del bucle de lectura/escritura Modbus TCP ($20\,\text{Hz}$), evitando bloqueos por latencia de red en el renderizado de video.
2. **Tolerancia a Fallos** (`MANO_CONECTADA`): El bloque `try-except` intenta importar el controlador `InspireHand` desde el módulo local `inspire_rh56dftp.py`. Si el módulo no se encuentra disponible, la bandera `MANO_CONECTADA` toma el valor `False`, permitiendo ejecutar y depurar toda la etapa de visión artificial y cinemática en **Modo Simulación**.
3. **Filtro de Consola**: Se suprimen las advertencias internas de depreciación de `google.protobuf` generadas por MediaPipe para mantener limpia la salida estándar de Jupyter Lab.

In [1]:
import pyrealsense2 as rs
import numpy as np
import cv2
import mediapipe as mp
import warnings
import csv
import os
import time
import threading

try:
    from inspire_rh56dftp import InspireHand
    MANO_CONECTADA = True
except ImportError:
    MANO_CONECTADA = False
    print("⚠ No se encontró 'inspire_rh56dftp.py'. El script correrá en modo Simulación.")

warnings.filterwarnings("ignore", category=UserWarning, module="google.protobuf.symbol_database")

## Bloque 2: Configuración de Red, Registros Modbus, Umbrales Anatómicos y Memoria Compartida

1. **Parámetros Operativos de Posición, Fuerza y Velocidad**:
   - Los vectores de 6 elementos siguen estrictamente el orden anatómico de los actuadores:
     $$\text{Índices DOF} = [\text{0: Meñique},\, \text{1: Anular},\, \text{2: Corazón},\, \text{3: Índice},\, \text{4: Flexión Pulgar},\, \text{5: Rotación Pulgar}]$$
   - `ABIERTO = [1000, 1000, 1000, 1000, 1000, 1000]`: Postura de extensión completa para los 6 DOF.
   - `FUERZA_SEGURA = [100, 100, 100, 100, 300, 300]`: Perfil de torque reducido inicial para desplazamientos libres y aproximación suave a objetos.
   - `FUERZA_MAXIMA = [1000, 1000, 1000, 1000, 1500, 1000]`: Techo de fuerza permitido durante la inyección incremental (hasta $1500$ en la flexión del pulgar y $1000$ en los demás actuadores).
   - `VELOCIDAD_OPERATIVA`: Vector uniforme construido a partir de `VELOCIDAD_GLOBAL_CONFIG = 1000`.

2. **Mapa de Registros Modbus TCP**:

| Constante en Código | Registro Modbus | Operación | Propósito Técnico |
| :--- | :--- | :--- | :--- |
| `REG_ANGLE_SET` | `1486` | Escritura | Inyección directa de la consigna de posición angular ($0 - 1000$). |
| `REG_FORCE_SET` | `1498` | Escritura | Configuración del límite de fuerza operativa ($0 - 1000 / 1500$). |
| `REG_SPEED_SET` | `1522` | Escritura | Configuración de la velocidad de desplazamiento ($0 - 1000$). |
| `REG_ANGLE_ACT` | `1546` | Lectura | Lectura en tiempo real de los sensores físicos de ángulo actual. |
| `REG_FORCE_ACT` | `1582` | Lectura | Lectura en tiempo real de los sensores físicos de fuerza actual. |

3. **Segmentación de Umbrales Táctiles y Cinemáticos**:
   - `UMBRALES_PRESION = np.array([10, 10, 10, 10, 50, 150])`: Sensibilidad táctil por actuador. Asigna `10` unidades a los cuatro dedos largos (`0:4`), `50` a la flexión del pulgar (`4`) para compensar la tensión mecánica de su resorte principal y `150` a la rotación del pulgar (`5`) para filtrar la fricción propia del riel de oposición.
   - `UMBRAL_POR_DEDO = np.array([100, 100, 100, 100, 15, 50])`: Banda muerta cinemática que exige cambios de $100$ unidades en los dedos principales, $15$ unidades en la flexión del pulgar y $50$ unidades en la rotación.

4. **Persistencia CSV y Variables Globales de Comunicación Inter-Hilos**:
   - Se garantizan mediante `os.makedirs(..., exist_ok=True)` las rutas hacia `gestos_guardados_3d.csv` y `rutinas_guardadas_3d.csv`.
   - Las variables con prefijo `VIS_*` transfieren las consignas y estados desde el hilo de visión hacia el hilo Modbus, mientras que las listas `MOD_*` retornan la telemetría física de la mano hacia el HUD visual sin bloqueos.

In [2]:
IP_MANO = "192.168.123.211"
ABIERTO = [1000, 1000, 1000, 1000, 1000, 1000]
FUERZA_SEGURA = [100, 100, 100, 100, 300, 300]
FUERZA_MAXIMA = [1000, 1000, 1000, 1000, 1500, 1000]
VELOCIDADES_MAXIMAS = [1000, 1000, 1000, 1000, 1000, 1000]

VELOCIDAD_GLOBAL_CONFIG = 1000
VELOCIDAD_OPERATIVA = [VELOCIDAD_GLOBAL_CONFIG] * 6

REG_ANGLE_SET = 1486  
REG_FORCE_SET = 1498  
REG_SPEED_SET = 1522  
REG_ANGLE_ACT = 1546  
REG_FORCE_ACT = 1582 
# Umbrales táctiles anatómicos
UMBRALES_PRESION = np.array([10, 10, 10, 10, 50, 150])
UMBRAL_POR_DEDO  = np.array([100, 100, 100, 100, 15, 50])

# Archivos .csv
DIRECTORIO_GUARDADO = os.path.expanduser("~/Documents/Jupyter/ManoRH56DFTP-2R/Gestos")
DIRECTORIO_RUTINAS = os.path.expanduser("~/Documents/Jupyter/ManoRH56DFTP-2R/Rutinas")
os.makedirs(DIRECTORIO_GUARDADO, exist_ok=True)
os.makedirs(DIRECTORIO_RUTINAS, exist_ok=True)
ARCHIVO_GESTOS_3D = os.path.join(DIRECTORIO_GUARDADO, "gestos_guardados_3d.csv")
ARCHIVO_RUTINAS_3D = os.path.join(DIRECTORIO_RUTINAS, "rutinas_guardadas_3d.csv")
CABECERAS_GESTOS = ["Nombre_Gesto", "Ang_Meñique", "Ang_Anular", "Ang_Corazón", "Ang_Índice", "Ang_Flexión_pulgar", "Ang_Rotación_pulgar"]
CABECERAS_RUTINAS = ["Nombre_Rutina", "Secuencia_Indices", "Secuencia_Nombres"]

# --- COMUNICACIÓN ENTRE HILOS ---
VIS_VEC_CALCULADO = ABIERTO.copy()
VIS_TRACKING      = False
VIS_MODO          = 1
VIS_REINICIAR     = False
VIS_SALIR         = False
VIS_GATILLO_FUERZA = False 

MOD_F_REALES = [0]*6
MOD_D_PAUSA  = [False]*6
MOD_O_DETECT = [False]*6
MOD_F_SET    = FUERZA_SEGURA.copy()

## Bloque 3: Lectura de Sensores (Complemento a Dos), Control Puntual y Gestión CSV

1. **Función** `leer_sensores_rapido(mano)`:
   - Consulta en bloque los 6 registros de posición real (`REG_ANGLE_ACT = 1546`) y los 6 registros de fuerza real (`REG_FORCE_ACT = 1582`).
   - **Corrección de Complemento a Dos ($16\,\text{bits}$)**: Debido a que los registros Modbus transmiten enteros sin signo (`uint16`), las oscilaciones negativas cercanas a cero se reportan como valores próximos a $65535$. A cada lectura $v$ se le aplica la transformación:
     $$v_{\text{corregido}} = \begin{cases} v, & \text{si } v < 32768 \\ v - 65536, & \text{si } v \ge 32768 \end{cases}$$
   - En las lecturas de fuerza se aplica adicionalmente $\max(0, v_{\text{corregido}})$ para truncar ruidos negativos a $0$.

2. **Función** `enviar_angulos(mano, angulos, timeout=5.0, tolerancia=40)`:
   - Utilizada para movimientos puntuales fuera del seguimiento continuo (apertura inicial, cierre del programa y ejecución de gestos/rutinas desde el menú).
   - Escribe primero `VELOCIDADES_MAXIMAS` en `REG_SPEED_SET` (`1522`), aguarda $10\,\text{ms}$ y envía la postura objetivo a `REG_ANGLE_SET` (`1486`). Posteriormente verifica cada $100\,\text{ms}$ mediante `leer_sensores_rapido(mano)` si el error absoluto en los 6 motores es menor o igual a $40$ unidades o hasta agotar el `timeout` de $5.0\,\text{s}$.

3. **Funciones de Base de Datos** (`cargar_gestos`, `cargar_rutinas`, `exportar_rutina`):
   - Gestionan la lectura y escritura sobre los archivos `.csv`, omitiendo de forma segura las cabeceras con `next(reader)` y formateando las secuencias de rutinas mediante índices (`"0,2,1"`) y nombres legibles (`"GestoA -> GestoC -> GestoB"`).

In [3]:
def leer_sensores_rapido(mano):
    if not MANO_CONECTADA or mano is None: return None, None
    try:
        angulos_raw = mano.read_holding(REG_ANGLE_ACT, 6)
        angulos = [v if v < 32768 else v - 65536 for v in angulos_raw]
        fuerzas_raw = mano.read_holding(REG_FORCE_ACT, 6)
        fuerzas = [max(0, v if v < 32768 else v - 65536) for v in fuerzas_raw]
        return np.array(angulos, dtype=int), np.array(fuerzas, dtype=int)
    except Exception: return None, None
        
def enviar_angulos(mano, angulos, timeout=5.0, tolerancia=40):
    if not MANO_CONECTADA or mano is None:
        time.sleep(1)
        return True
    try:
        mano._write_and_ack(REG_SPEED_SET, VELOCIDADES_MAXIMAS)
        time.sleep(0.01)
        mano._write_and_ack(REG_ANGLE_SET, angulos)
    except Exception: pass
        
    inicio = time.time()
    while time.time() - inicio < timeout:
        try:
            angulos_act, _ = leer_sensores_rapido(mano)
            if angulos_act is not None and all(abs(act - obj) <= tolerancia for act, obj in zip(angulos_act, angulos)):
                return True
        except Exception: pass 
        time.sleep(0.1)
    return False

def cargar_gestos():
    if not os.path.exists(ARCHIVO_GESTOS_3D): return []
    with open(ARCHIVO_GESTOS_3D, mode='r') as file:
        reader = csv.reader(file)
        try: next(reader) 
        except StopIteration: pass
        return [row for row in reader if row]

def cargar_rutinas():
    if not os.path.exists(ARCHIVO_RUTINAS_3D): return []
    with open(ARCHIVO_RUTINAS_3D, mode='r') as file:
        reader = csv.reader(file)
        try: next(reader) 
        except StopIteration: pass
        return [row for row in reader if row and len(row) >= 2]

def exportar_rutina(nombre, indices, gestos_guardados):
    existe = os.path.exists(ARCHIVO_RUTINAS_3D)
    sec_idx = ",".join(str(i) for i in indices)
    sec_nom = " -> ".join([gestos_guardados[i][0] for i in indices])
    with open(ARCHIVO_RUTINAS_3D, mode='a', newline='') as file:
        writer = csv.writer(file)
        if not existe: writer.writerow(CABECERAS_RUTINAS)
        writer.writerow([nombre, sec_idx, sec_nom])
    print(f" ✓ Rutina '{nombre}' guardada.")

## Bloque 4: Hilo Secundario (`hilo_control_modbus`) — Bucle Cerrado a 20 Hz y 3 Modos de Telecontrol

Este hilo se ejecuta en paralelo al hilo de visión con un periodo estricto de `FRECUENCIA_ENVIO = 0.05` ($20\,\text{Hz}$). En cada iteración lee las consignas de `VIS_VEC_CALCULADO`, consulta los sensores reales mediante `leer_sensores_rapido(mano)` y aplica el modo de operación activo (`VIS_MODO`):

1. **Inyección Directa de Velocidad y Ángulo**:
   - Antes de escribir un nuevo vector de posición en `REG_ANGLE_SET` (`1486`), el hilo inyecta `VELOCIDAD_OPERATIVA` en `REG_SPEED_SET` (`1522`) con una micro-pausa de $10\,\text{ms}$ (`time.sleep(0.01)`), garantizando que el controlador interno respete el límite de velocidad en cada actualización.

2. **Modo 0 — Teleoperación Estándar** (`modo_actual == 0`):
   - Mantiene `FUERZA_SEGURA` en todos los motores y envía `vector_objetivo` cada vez que algún dedo supera su umbral en `UMBRAL_POR_DEDO`.

3. **Modo 1 — Detección de Presión con Pausa en Posición de $0.5\,\text{s}$** (`modo_actual == 1`):
   - **Freno Instantáneo por Contacto**: Si un dedo libre (`not dedo_pausado[i]`) registra una fuerza real $F_{\text{real}, i} \ge \text{UMBRALES\_PRESION}_i$ mientras intenta cerrarse (`vector_objetivo[i] < ang_reales[i] + 20`), el sistema enclava `dedo_pausado[i] = True`, captura el ángulo físico instantáneo en `angulo_pausa[i]`, registra la consigna de la cámara en `consigna_al_pausar[i]` e inicia el temporizador `tiempo_inicio_pausa[i]`. El dedo se bloquea en `angulo_pausa[i]` sin regresar a la posición abierta.
   - **Pausa Obligatoria de $0.5\,\text{s}$**: Durante los primeros $0.5\,\text{s}$ (`PAUSA_OBLIGATORIA_S`) posteriores al contacto, cualquier comando de movimiento sobre ese dedo es ignorado.
   - **Condición Dual de Desbloqueo**: Cumplidos los $0.5\,\text{s}$, el dedo permanece pausado hasta que el usuario genere una **nueva señal de movimiento** ($\vert{}\text{vector\_vision}_i - \text{consigna\_al\_pausar}_i\vert{} \ge \text{UMBRAL\_POR\_DEDO}_i$) **y** se cumpla que el movimiento es de **apertura** (`quiere_abrir`) o que la lectura de presión cayó por debajo del umbral (`presion_libre`).

4. **Modo 2 — Fuerza Incremental por Gatillo de Teclado** (`modo_actual == 2`):
   - **Detección Inicial Segura**: La mano se mueve con `FUERZA_SEGURA`. Al detectar un objeto ($F_{\text{real}, i} \ge \text{UMBRALES\_PRESION}_i$), activa la bandera `objeto_detectado[i] = True` para ese dedo.
   - **Rampa Progresiva con Gatillo** (`'f'`): Para todos los dedos con `objeto_detectado[i] == True`, mientras el operador mantenga presionada la tecla `'f'` (`gatillo_fuerza == True`), el límite de fuerza `fuerza_incremental_cfg[i]` se incrementa en `PASO_INCREMENTO_FUERZA = 30` unidades por ciclo ($+600\,\text{unidades/s}$) hasta alcanzar `FUERZA_MAXIMA[i]`, actualizando `REG_FORCE_SET` (`1498`).
   - **Micro-Oscilación de Empuje ($\pm 1$)**: Dado que el firmware detiene el motor cuando alcanza el límite de fuerza inicial, se suma `oscilador_empuje` (alternando entre $0$ y $1$) a `vector_objetivo[i]` en los dedos en contacto, obligando al controlador a aplicar inmediatamente el nuevo torque sobre el objeto.
   - **Restablecimiento Automático**: Al abrir el dedo (`quiere_soltar`) y desaparecer la presión física, `objeto_detectado[i]` pasa a `False` y su fuerza retorna a `FUERZA_SEGURA[i]`.

In [4]:
def hilo_control_modbus(mano):
    global VIS_REINICIAR, MOD_F_REALES, MOD_D_PAUSA, MOD_O_DETECT, MOD_F_SET
    FRECUENCIA_ENVIO = 0.05
    ultimo_vector_enviado = np.array(ABIERTO, dtype=int)
    
    PAUSA_OBLIGATORIA_S = 0.5
    dedo_pausado = np.zeros(6, dtype=bool)
    angulo_pausa = np.array(ABIERTO, dtype=int)
    consigna_al_pausar = np.array(ABIERTO, dtype=int)
    tiempo_inicio_pausa = np.zeros(6, dtype=float)

    PASO_INCREMENTO_FUERZA = 30
    objeto_detectado = np.zeros(6, dtype=bool)
    fuerza_incremental_cfg = np.array(FUERZA_SEGURA, dtype=int)
    ultima_fuerza_enviada = np.array(FUERZA_SEGURA, dtype=int)
    oscilador_empuje = 0
    
    if MANO_CONECTADA and mano is not None:
        try:
            mano._write_and_ack(REG_FORCE_SET, FUERZA_SEGURA)
            time.sleep(0.05)
            mano._write_and_ack(REG_SPEED_SET, VELOCIDAD_OPERATIVA)
        except Exception: pass

    while not VIS_SALIR:
        inicio_ciclo = time.time()

        if VIS_REINICIAR:
            dedo_pausado[:] = False
            objeto_detectado[:] = False
            fuerza_incremental_cfg = np.array(FUERZA_SEGURA, dtype=int)
            if MANO_CONECTADA and mano is not None:
                try:
                    mano._write_and_ack(REG_FORCE_SET, FUERZA_SEGURA)
                    time.sleep(0.05)
                    mano._write_and_ack(REG_SPEED_SET, VELOCIDAD_OPERATIVA)
                except Exception: pass
            ultima_fuerza_enviada = np.array(FUERZA_SEGURA, dtype=int)
            VIS_REINICIAR = False

        if not VIS_TRACKING or not MANO_CONECTADA or mano is None:
            time.sleep(0.05)
            continue

        modo_actual   = VIS_MODO
        vector_vision = VIS_VEC_CALCULADO
        gatillo_fuerza = VIS_GATILLO_FUERZA 

        ang_reales, f_reales = leer_sensores_rapido(mano)
        if ang_reales is not None and f_reales is not None:
            MOD_F_REALES = f_reales.tolist()
        else:
            ang_reales = ultimo_vector_enviado.copy()
            f_reales = np.zeros(6, dtype=int)

        vector_objetivo = ultimo_vector_enviado.copy()
        
        for i in range(6):
            umbral_i = 1 if i == 5 else UMBRAL_POR_DEDO[i]
            if abs(vector_vision[i] - ultimo_vector_enviado[i]) >= umbral_i:
                vector_objetivo[i] = vector_vision[i]

        if modo_actual == 0:  
            if not np.array_equal(vector_objetivo, ultimo_vector_enviado):
                try:
                    mano._write_and_ack(REG_SPEED_SET, VELOCIDAD_OPERATIVA)
                    time.sleep(0.01)
                    mano._write_and_ack(REG_ANGLE_SET, vector_objetivo.tolist())
                except Exception: pass
                ultimo_vector_enviado = vector_objetivo.copy()

        elif modo_actual == 1: 
            forzado_por_freno = False
            for i in range(6):
                if not dedo_pausado[i]:
                    if f_reales[i] >= UMBRALES_PRESION[i] and vector_objetivo[i] < ang_reales[i] + 20:
                        dedo_pausado[i] = True
                        angulo_pausa[i] = int(np.clip(ang_reales[i], 0, 1000))
                        consigna_al_pausar[i] = vector_vision[i]
                        tiempo_inicio_pausa[i] = inicio_ciclo
                        vector_objetivo[i] = angulo_pausa[i]
                        forzado_por_freno = True
                else:
                    if (inicio_ciclo - tiempo_inicio_pausa[i]) < PAUSA_OBLIGATORIA_S:
                        vector_objetivo[i] = angulo_pausa[i]
                    else:
                        hay_nueva_senal = abs(vector_vision[i] - consigna_al_pausar[i]) >= UMBRAL_POR_DEDO[i]
                        quiere_abrir = vector_vision[i] > (angulo_pausa[i] + UMBRAL_POR_DEDO[i])
                        presion_libre = f_reales[i] < UMBRALES_PRESION[i]
                        
                        if hay_nueva_senal and (quiere_abrir or presion_libre):
                            dedo_pausado[i] = False
                            vector_objetivo[i] = vector_vision[i]
                        else:
                            vector_objetivo[i] = angulo_pausa[i]

            if forzado_por_freno or not np.array_equal(vector_objetivo, ultimo_vector_enviado):
                try:
                    mano._write_and_ack(REG_SPEED_SET, VELOCIDAD_OPERATIVA)
                    time.sleep(0.01)
                    mano._write_and_ack(REG_ANGLE_SET, vector_objetivo.tolist())
                except Exception: pass
                ultimo_vector_enviado = vector_objetivo.copy()
            MOD_D_PAUSA = dedo_pausado.tolist()

        elif modo_actual == 2: 
            oscilador_empuje = 1 - oscilador_empuje
            alguno_incrementando = False

            for i in range(6):
                quiere_soltar = (vector_vision[i] > ang_reales[i] + 80) or (vector_vision[i] >= 920)
                if quiere_soltar and f_reales[i] < UMBRALES_PRESION[i]:
                    objeto_detectado[i] = False
                    fuerza_incremental_cfg[i] = FUERZA_SEGURA[i]
                elif f_reales[i] >= UMBRALES_PRESION[i]:
                    objeto_detectado[i] = True

                if objeto_detectado[i]:
                    if gatillo_fuerza:
                        if fuerza_incremental_cfg[i] < FUERZA_MAXIMA[i]:
                            fuerza_incremental_cfg[i] = min(FUERZA_MAXIMA[i], fuerza_incremental_cfg[i] + PASO_INCREMENTO_FUERZA)
                            alguno_incrementando = True
                    vector_objetivo[i] = int(np.clip(vector_objetivo[i] + oscilador_empuje, 0, 1000))

            if not np.array_equal(fuerza_incremental_cfg, ultima_fuerza_enviada):
                try:
                    mano._write_and_ack(REG_FORCE_SET, fuerza_incremental_cfg.tolist())
                except Exception: pass
                ultima_fuerza_enviada = fuerza_incremental_cfg.copy()

            if alguno_incrementando or not np.array_equal(vector_objetivo, ultimo_vector_enviado):
                try:
                    mano._write_and_ack(REG_SPEED_SET, VELOCIDAD_OPERATIVA)
                    time.sleep(0.01)
                    mano._write_and_ack(REG_ANGLE_SET, vector_objetivo.tolist())
                except Exception: pass
                ultimo_vector_enviado = vector_objetivo.copy()
            
            MOD_O_DETECT = objeto_detectado.tolist()
            MOD_F_SET    = fuerza_incremental_cfg.tolist()

        tiempo_transcurrido = time.time() - inicio_ciclo
        if tiempo_transcurrido < FRECUENCIA_ENVIO:
            time.sleep(FRECUENCIA_ENVIO - tiempo_transcurrido)

## Bloque 5: Calibración Biométrica 3D en 4 Fases (`ejecutar_calibracion`)

### 1. Extracción de Nodos, Renderizado Independiente y Deproyección 3D
El sistema selecciona 11 nodos anatómicos del modelo de MediaPipe: `nodos_interes = [0, 2, 4, 5, 8, 9, 12, 13, 16, 17, 20]` cuando detecta la etiqueta `"Left"` (mano derecha física frente a cámara sin efecto espejo):
- **Nodos de falanges principales**: Base y punta del Índice (`5` y `8`), Corazón (`9` y `12`), Anular (`13` y `16`) y Meñique (`17` y `20`).
- **Nodos del pulgar y palma**: Muñeca (`0`), base del pulgar (`2`) y yema del pulgar (`4`).
- **Renderizado Independiente de la Profundidad**: El trazado del *wireframe* sobre `color_image` (líneas grises para la estructura de palma/muñeca y líneas verdes/amarillas para las distancias operativas) se ejecuta directamente a partir del diccionario 2D `pixeles_dibujo` antes de evaluar `if prof_valida:`. Esto garantiza que el esqueleto visual permanezca estable en pantalla sin parpadeos.
- **Deproyección Métrica**: Cuando `z_metros = depth_frame.get_distance(cx, cy)` es mayor a cero en los 11 nodos, `rs.rs2_deproject_pixel_to_point(intrinsics, [cx, cy], z_metros)` transforma cada píxel 2D en una coordenada física tridimensional $\mathbf{P}_k = [X_k, Y_k, Z_k]$ en metros.

### 2. Vector de Magnitudes Físicas (`D_actual`)
Con los 11 puntos 3D validados, se construye el vector `D_actual` de 6 elementos:
$$\text{D\_actual} = \begin{bmatrix} \Vert{}\mathbf{P}_{20} - \mathbf{P}_{17}\Vert{}_2 \\ \Vert{}\mathbf{P}_{16} - \mathbf{P}_{13}\Vert{}_2 \\ \Vert{}\mathbf{P}_{12} - \mathbf{P}_{9}\Vert{}_2 \\ \Vert{}\mathbf{P}_{8} - \mathbf{P}_{5}\Vert{}_2 \\ \Vert{}\mathbf{P}_{4} - \mathbf{P}_{5}\Vert{}_2 \\ \vert{}Z_4 - Z_5\vert{} \end{bmatrix} \begin{matrix} \leftarrow \text{pos(0): Meñique} \\ \leftarrow \text{pos(1): Anular} \\ \leftarrow \text{pos(2): Corazón} \\ \leftarrow \text{pos(3): Índice} \\ \leftarrow \text{pos(4): Flexión Pulgar} \\ \leftarrow \text{pos(5): Rotación Pulgar} \end{matrix}$$
- **Motores `0` a `4`**: Norma euclidiana tridimensional $\Vert{}\mathbf{P}_a - \mathbf{P}_b\Vert{}_2 = \sqrt{\Delta X^2 + \Delta Y^2 + \Delta Z^2}$.
- **Motor `5` (Rotación del Pulgar)**: Valor absoluto de la diferencia de profundidad en el eje $Z$ entre la yema del pulgar (nodo `4`) y la base del índice (nodo `5`): $\Delta Z = \vert{}Z_4 - Z_5\vert{}$.

### 3. Máquina de Estados de las 4 Fases de Calibración
Cada fase captura 3 muestras al presionar `'c'` y almacena su promedio (`np.mean(..., axis=0)`):
1. **Fase 1** (`MÁXIMA` **— Mano Abierta**): Guarda la extensión máxima en `D_max[0:5]` y la mínima diferencia de profundidad $\Delta Z$ en `D_min[5]` (correspondiente a la rotación abierta `1000`).
2. **Fase 2** (`MÍNIMA_DEDOS` **— Puño 4 Dedos)**: Registra las distancias mínimas de cierre en `D_min[0:4]`.
3. **Fase 3** (`MÍNIMA_FLEX_PULGAR` **— Flexión Pulgar)**: Acercando el pulgar al índice, registra la distancia mínima entre los nodos `4` y `5` en `D_min[4]`.
4. **Fase 4** (`MÍNIMA_ROT_PULGAR` **— Rotación Pulgar $Z$**): Rotando el pulgar en oposición para generar la máxima separación de profundidad $\Delta Z$ con la base del índice, guarda el resultado en `D_max[5]` (correspondiente al cierre de rotación `0`).

In [5]:
def ejecutar_calibracion(pipeline, align, hands, intrinsics):
    modo = "MÁXIMA"
    muestras_max, muestras_min_dedos, muestras_min_flex_p, muestras_min_rot_p = [], [], [], []
    D_max, D_min = np.zeros(6), np.zeros(6)
    
    while True:
        frames = pipeline.wait_for_frames()
        aligned_frames = align.process(frames)
        color_frame = aligned_frames.get_color_frame()
        depth_frame = aligned_frames.get_depth_frame()
        if not color_frame or not depth_frame: continue
        
        color_image = np.asanyarray(color_frame.get_data())
        image_rgb = cv2.cvtColor(color_image, cv2.COLOR_BGR2RGB)
        
        results = hands.process(image_rgb)
        
        D_actual = np.zeros(6)
        mano_detectada = False
        prof_valida = True

        if results.multi_hand_landmarks and results.multi_handedness:
            for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                if handedness.classification[0].label == "Left":
                    mano_detectada = True
                    h, w, _ = color_image.shape
                    nodos_interes = [0, 2, 4, 5, 8, 9, 12, 13, 16, 17, 20]
                    puntos_3d, pixeles_dibujo = {}, {}
                    
                    for idx in nodos_interes:
                        lm = hand_landmarks.landmark[idx]
                        cx, cy = int(lm.x * w), int(lm.y * h)
                        cx, cy = max(0, min(cx, w - 1)), max(0, min(cy, h - 1))
                        pixeles_dibujo[idx] = (cx, cy)
                        
                        z_metros = depth_frame.get_distance(cx, cy)
                        
                        if z_metros > 0:
                            puntos_3d[idx] = np.array(rs.rs2_deproject_pixel_to_point(intrinsics, [cx, cy], z_metros))
                        else: 
                            prof_valida = False

                    # --- DIBUJO DEL WIREFRAME INDEPENDIENTE DE LA PROFUNDIDAD ---
                    for idx, (cx, cy) in pixeles_dibujo.items():
                        cv2.circle(color_image, (cx, cy), 6, (255, 0, 0), cv2.FILLED)
                    
                    if len(pixeles_dibujo) == 11:
                        cv2.line(color_image, pixeles_dibujo[0], pixeles_dibujo[5], (200, 200, 200), 2)
                        cv2.line(color_image, pixeles_dibujo[0], pixeles_dibujo[9], (200, 200, 200), 2)
                        cv2.line(color_image, pixeles_dibujo[0], pixeles_dibujo[13], (200, 200, 200), 2)
                        cv2.line(color_image, pixeles_dibujo[0], pixeles_dibujo[17], (200, 200, 200), 2)
                        cv2.line(color_image, pixeles_dibujo[0], pixeles_dibujo[2], (200, 200, 200), 2)
                        cv2.line(color_image, pixeles_dibujo[2], pixeles_dibujo[4], (200, 200, 200), 2)
                        cv2.line(color_image, pixeles_dibujo[5], pixeles_dibujo[8], (0, 255, 0), 3)    
                        cv2.line(color_image, pixeles_dibujo[9], pixeles_dibujo[12], (0, 255, 0), 3)   
                        cv2.line(color_image, pixeles_dibujo[13], pixeles_dibujo[16], (0, 255, 0), 3)  
                        cv2.line(color_image, pixeles_dibujo[17], pixeles_dibujo[20], (0, 255, 0), 3)  
                        cv2.line(color_image, pixeles_dibujo[5], pixeles_dibujo[4], (0, 255, 255), 3)  

                    # --- MATEMÁTICA Y LÓGICA PROTEGIDAS ---
                    if prof_valida:
                        D_actual = np.array([
                            np.linalg.norm(puntos_3d[20] - puntos_3d[17]),
                            np.linalg.norm(puntos_3d[16] - puntos_3d[13]),
                            np.linalg.norm(puntos_3d[12] - puntos_3d[9]),
                            np.linalg.norm(puntos_3d[8] - puntos_3d[5]),
                            np.linalg.norm(puntos_3d[4] - puntos_3d[5]),
                            abs(puntos_3d[4][2] - puntos_3d[5][2])
                        ])
                    else:
                        cv2.putText(color_image, "Mueve la mano, perdiendo profundidad", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

                    if modo == "MÁXIMA":
                        cv2.putText(color_image, f"1. Abierta ({len(muestras_max)}/3) - 'c'", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                    elif modo == "MÍNIMA_DEDOS":
                        cv2.putText(color_image, f"2. Puno 4 Dedos ({len(muestras_min_dedos)}/3) - 'c'", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                    elif modo == "MÍNIMA_FLEX_PULGAR":
                        cv2.putText(color_image, f"3. Flexion Pulgar ({len(muestras_min_flex_p)}/3) - 'c'", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
                    elif modo == "MÍNIMA_ROT_PULGAR":
                        cv2.putText(color_image, f"4. Rotacion Pulgar Z ({len(muestras_min_rot_p)}/3) - 'c'", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 255), 2)

        cv2.imshow('Calibracion 3D', color_image)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('c') and mano_detectada and prof_valida:
            if modo == "MÁXIMA":
                muestras_max.append(D_actual.copy())
                if len(muestras_max) == 3:
                    prom = np.mean(np.array(muestras_max), axis=0)
                    D_max[0:5] = prom[0:5]
                    D_min[5] = prom[5]
                    modo = "MÍNIMA_DEDOS"
            elif modo == "MÍNIMA_DEDOS":
                muestras_min_dedos.append(D_actual.copy())
                if len(muestras_min_dedos) == 3:
                    D_min[0:4] = np.mean(np.array(muestras_min_dedos), axis=0)[0:4]
                    modo = "MÍNIMA_FLEX_PULGAR"
            elif modo == "MÍNIMA_FLEX_PULGAR":
                muestras_min_flex_p.append(D_actual.copy())
                if len(muestras_min_flex_p) == 3:
                    D_min[4] = np.mean(np.array(muestras_min_flex_p), axis=0)[4]
                    modo = "MÍNIMA_ROT_PULGAR"
            elif modo == "MÍNIMA_ROT_PULGAR":
                muestras_min_rot_p.append(D_actual.copy())
                if len(muestras_min_rot_p) == 3:
                    D_max[5] = np.mean(np.array(muestras_min_rot_p), axis=0)[5]
                    cv2.destroyWindow('Calibracion 3D')
                    return D_max, D_min
        elif key == ord('q'):
            cv2.destroyWindow('Calibracion 3D')
            return None, None

## Bloque 6: Programa Principal (`main`) — Inicialización y Lanzamiento de Hilos

- **Inicialización de Hardware y Cámara**: Conecta con la mano en `IP_MANO`, escribe `FUERZA_SEGURA` en `REG_FORCE_SET` (`1498`) y posiciona los actuadores en `ABIERTO`. Enseguida inicializa MediaPipe (`max_num_hands=1`, `min_detection_confidence=0.7`) y levanta el flujo de la cámara RealSense a $640 \times 480$ píxeles y $30\,\text{FPS}$, alineando el mapa de profundidad al flujo RGB (`rs.align(rs.stream.color)`).
- **Matrices Predeterminadas** (`D_max` **y** `D_min`): Permiten iniciar la operación directamente con la Opción 1 o actualizarlas en RAM ejecutando la Opción 2 (`ejecutar_calibracion`).
- **Arranque del Hilo Modbus**: Se instancia `threading.Thread(target=hilo_control_modbus, args=(mano,), daemon=True)` y se inicia con `hilo_modbus.start()`, delegando toda la comunicación TCP/IP al hilo secundario mientras el hilo principal mantiene los $30\,\text{FPS}$ constantes.

---

## Bloque 7: Bucle de Seguimiento Continuo, Normalización e Inversión Cinemática

Dentro del bucle `while True:`, cuando los 11 nodos cuentan con lectura de profundidad válida (`if prof_valida:`), se ejecuta el bloque matemático central:
```bash
denominador = np.where(np.abs(D_max - D_min) < 1e-6, 1e-6, D_max - D_min)
Apertura_decimal = np.clip((D_actual - D_min) / denominador, 0.0, 1.0)
Apertura_decimal[5] = 1.0 - Apertura_decimal[5]
vector_crudo = (Apertura_decimal * 1000).astype(int)
```
1. **Protección contra División por Cero**: `np.where(np.abs(D_max - D_min) < 1e-6, 1e-6, D_max - D_min)` asegura la estabilidad numérica del cociente.
2. **Normalización Lineal Acotada**: Se proyecta `D_actual` al intervalo $[0.0, 1.0]$ mediante:
   $$\text{Apertura}_i = \text{clip}\left(\frac{D_{\text{actual}, i} - D_{\min, i}}{D_{\max, i} - D_{\min, i}},\, 0.0,\, 1.0\right)$$
3. **Inversión del Motor de Rotación** (`pos(5)`): Como la máxima separación en profundidad ($\Delta Z \to D_{\max}[5]$) equivale al cierre de rotación (`0`) y la mínima separación ($\Delta Z \to D_{\min}[5]$) equivale a la apertura (`1000`), se invierte el quinto índice:
   $$\text{Apertura}_5 \leftarrow 1.0 - \text{Apertura}_5$$
4. **Escalado Entero**: Se multiplica por $1000$ y se convierte a entero (`vector_crudo`) para obtener el rango $[0, 1000]$ nativo del controlador.

---

## Bloque 8: Máquina de Estados de Protección para la Rotación del Pulgar
```bash
indice_actual, flex_actual, rot_calculada = vector_crudo[3], vector_crudo[4], vector_crudo[5]
delta_flex = flex_actual - flexion_anterior

if indice_actual <= 10:
    rotacion_retenida, rotacion_bloqueada = 1000, True
else:
    if rotacion_retenida <= 150 or rotacion_bloqueada:
        if delta_flex < -15: rotacion_bloqueada = True
        elif delta_flex > 15: rotacion_bloqueada = False
    else: rotacion_bloqueada = False

if abs(delta_flex) > 15: flexion_anterior = flex_actual
if not rotacion_bloqueada and abs(rot_calculada - rotacion_retenida) >= 50:
    rotacion_retenida = rot_calculada

vector_crudo[5] = rotacion_retenida
VIS_VEC_CALCULADO = vector_crudo.copy()
```
1. **Prioridad 1 — Oclusión por Cierre del Dedo Índice** (`indice_actual <= 10`): Cuando el dedo índice se cierra por completo, obstruye la visión de la cámara hacia el nodo `5`, alterando la lectura de $Z_5$. El sistema fuerza `rotacion_retenida = 1000` y activa `rotacion_bloqueada = True` para prevenir colisiones mecánicas.
2. **Prioridad 2 — Bloqueo por Flexión en Oposición** (`rotacion_retenida <= 150`): Cuando el pulgar ha rotado hacia la palma y comienza a flexionarse (`delta_flex < -15`), se activa `rotacion_bloqueada = True`, congelando la rotación mientras el pulgar desciende. Al extender nuevamente el pulgar (`delta_flex > 15`), la bandera se libera (`rotacion_bloqueada = False`).
3. **Filtro de Escalón** (`>= 50`) **y Publicación**: Si la rotación no está bloqueada, `rotacion_retenida` se actualiza cuando la variación iguala o supera $50$ unidades, publicando finalmente el vector en `VIS_VEC_CALCULADO` para su consumo inmediato por el hilo secundario.
---
## Bloque 9: Renderizado Visual (HUD Multimodo), Gatillo por Teclado, Menú de Rutinas y Cierre Seguro

- **HUD de Telemetría por Modo**:
  - Muestra el estado de seguimiento (`EN VIVO` / `PAUSADO`) y el modo activo (`0: ESTANDAR`, `1: PAUSA PRESION`, `2: FUERZA INCREMENTAL`).
  - En el **Modo 1**, despliega en vivo el estado de cada dedo (`P` = Pausado, `L` = Libre) desde `MOD_D_PAUSA` y las fuerzas físicas medidas `MOD_F_REALES`.
  - En el **Modo 2**, indica el estado del gatillo `'f'` (`GATILLO ACTIVO (+F)` / `GATILLO SUELTO (BLOQ)`), el vector de fuerzas configuradas `MOD_F_SET` y las banderas binarias de contacto por dedo `MOD_O_DETECT`.
- **Controles por Teclado** (`cv2.waitKey(1)`):
  - **`'f'` (Gatillo de Fuerza Incremental)**: Activa `VIS_GATILLO_FUERZA = True` mientras se mantiene presionada la tecla para incrementar progresivamente el torque en el Modo 2; al soltarla regresa a `False`.
  - **`'e'` (Toggle Tracking)**: Conmuta `VIS_TRACKING` para iniciar o pausar el envío de comandos en el hilo Modbus.
  - **`'m'` (Selector Cíclico de Modo)**: Alterna `VIS_MODO` entre `0`, `1` y `2`, activando `VIS_REINICIAR = True` para que el hilo secundario limpie estados y restablezca `FUERZA_SEGURA`.
  - **`'s'` y `'r'` (Guardado de Gestos y Menú de Rutinas 3D)**: Pausan el seguimiento y habilitan por consola el almacenamiento de posturas o la creación, exportación y reproducción de secuencias desde archivos `.csv`.
  - **`'q'` y Bloque `finally`**: Activa `VIS_SALIR = True`, detiene la cámara RealSense (`pipeline.stop()`), sincroniza el cierre del hilo secundario mediante `hilo_modbus.join()`, restaura `FUERZA_SEGURA` y `VELOCIDADES_MAXIMAS`, envía la mano a `ABIERTO` y cierra el socket TCP (`mano.close()`).

In [6]:
def main():
    global VIS_VEC_CALCULADO, VIS_TRACKING, VIS_MODO, VIS_REINICIAR, VIS_SALIR, VIS_GATILLO_FUERZA
    
    mano = None
    if MANO_CONECTADA:
        try:
            mano = InspireHand(ip=IP_MANO)
            mano._write_and_ack(REG_FORCE_SET, FUERZA_SEGURA)
            time.sleep(0.1)
            enviar_angulos(mano, ABIERTO)
        except Exception as e:
            print(f"Error conectando a la mano: {e}. Pasando a simulación.")

    D_max = np.array([0.1045, 0.1245, 0.1312, 0.1197, 0.1239, 0.0650])
    D_min = np.array([0.0261, 0.0280, 0.0250, 0.0218, 0.0474, 0.0150]) 
            
    mp_hands = mp.solutions.hands
    hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7)

    pipeline = rs.pipeline()
    config = rs.config()
    config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, 30)
    config.enable_stream(rs.stream.depth, 640, 480, rs.format.z16, 30)
    align = rs.align(rs.stream.color)
    pipeline.start(config)

    perfil_camara = pipeline.get_active_profile()
    intrinsics = perfil_camara.get_stream(rs.stream.depth).as_video_stream_profile().get_intrinsics()

    opcion = input("1. Usar calibración predeterminada\n2. Calibrar mano (Recomendado)\nSeleccione (1 o 2): ").strip()
    if opcion == '2':
        D_max_nuevo, D_min_nuevo = ejecutar_calibracion(pipeline, align, hands, intrinsics)
        if D_max_nuevo is not None:
            D_max, D_min = D_max_nuevo, D_min_nuevo

    print(f"\n=== MODO ACTIVO | VELOCIDAD LIMITADA A {VELOCIDAD_GLOBAL_CONFIG} ===")
    print(" 'e' : Iniciar / Pausar Seguimiento")
    print(" 'm' : Cambiar Modo (0: Estándar | 1: Pausa 0.5s | 2: Fuerza Incremental)")
    print(" 'f' : MANTENER PRESIONADO para inyectar Fuerza en Modo 2 (Gatillo)")
    print(" 's' : Guardar Gesto | 'r' : Menú Rutinas | 'q' : Salir")

    MODOS_TELECONTROL = ["0: ESTANDAR", "1: PAUSA PRESION", "2: FUERZA INCREMENTAL"]
    
    hilo_modbus = threading.Thread(target=hilo_control_modbus, args=(mano,), daemon=True)
    hilo_modbus.start()
    
    rotacion_retenida = 1000
    flexion_anterior = 1000
    rotacion_bloqueada = False

    try:
        while True:
            frames = pipeline.wait_for_frames()
            aligned_frames = align.process(frames)
            color_frame = aligned_frames.get_color_frame()
            depth_frame = aligned_frames.get_depth_frame()
            if not color_frame or not depth_frame: continue
            
            color_image = np.asanyarray(color_frame.get_data())
            image_rgb = cv2.cvtColor(color_image, cv2.COLOR_BGR2RGB)
            
            results = hands.process(image_rgb)

            if results.multi_hand_landmarks and results.multi_handedness:
                h, w, _ = color_image.shape
                for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                    
                    if handedness.classification[0].label == "Left": 
                        nodos_interes = [0, 2, 4, 5, 8, 9, 12, 13, 16, 17, 20]
                        puntos_3d, pixeles_dibujo = {}, {}
                        prof_valida = True
                        
                        for idx in nodos_interes:
                            lm = hand_landmarks.landmark[idx]
                            cx, cy = int(lm.x * w), int(lm.y * h)
                            cx, cy = max(0, min(cx, w - 1)), max(0, min(cy, h - 1))
                            pixeles_dibujo[idx] = (cx, cy)
                            
                            z_metros = depth_frame.get_distance(cx, cy)
                            
                            if z_metros > 0:
                                puntos_3d[idx] = np.array(rs.rs2_deproject_pixel_to_point(intrinsics, [cx, cy], z_metros))
                            else: 
                                prof_valida = False

                        # --- DIBUJO DEL WIREFRAME INDEPENDIENTE DE LA PROFUNDIDAD ---
                        for idx, (cx, cy) in pixeles_dibujo.items():
                            cv2.circle(color_image, (cx, cy), 6, (255, 0, 0), cv2.FILLED)
                        
                        if len(pixeles_dibujo) == 11:
                            cv2.line(color_image, pixeles_dibujo[0], pixeles_dibujo[5], (200, 200, 200), 2)
                            cv2.line(color_image, pixeles_dibujo[0], pixeles_dibujo[9], (200, 200, 200), 2)
                            cv2.line(color_image, pixeles_dibujo[0], pixeles_dibujo[13], (200, 200, 200), 2)
                            cv2.line(color_image, pixeles_dibujo[0], pixeles_dibujo[17], (200, 200, 200), 2)
                            cv2.line(color_image, pixeles_dibujo[0], pixeles_dibujo[2], (200, 200, 200), 2)
                            cv2.line(color_image, pixeles_dibujo[2], pixeles_dibujo[4], (200, 200, 200), 2)
                            cv2.line(color_image, pixeles_dibujo[5], pixeles_dibujo[8], (0, 255, 0), 3)    
                            cv2.line(color_image, pixeles_dibujo[9], pixeles_dibujo[12], (0, 255, 0), 3)   
                            cv2.line(color_image, pixeles_dibujo[13], pixeles_dibujo[16], (0, 255, 0), 3)  
                            cv2.line(color_image, pixeles_dibujo[17], pixeles_dibujo[20], (0, 255, 0), 3)  
                            cv2.line(color_image, pixeles_dibujo[5], pixeles_dibujo[4], (0, 255, 255), 3)  

                        # --- MATEMÁTICA Y LÓGICA PROTEGIDAS ---
                        if prof_valida:
                            D_actual = np.array([
                                np.linalg.norm(puntos_3d[20] - puntos_3d[17]),
                                np.linalg.norm(puntos_3d[16] - puntos_3d[13]),
                                np.linalg.norm(puntos_3d[12] - puntos_3d[9]),
                                np.linalg.norm(puntos_3d[8] - puntos_3d[5]),
                                np.linalg.norm(puntos_3d[4] - puntos_3d[5]),
                                abs(puntos_3d[4][2] - puntos_3d[5][2])
                            ])
                            
                            denominador = np.where(np.abs(D_max - D_min) < 1e-6, 1e-6, D_max - D_min)
                            Apertura_decimal = np.clip((D_actual - D_min) / denominador, 0.0, 1.0)
                            Apertura_decimal[5] = 1.0 - Apertura_decimal[5]
                            vector_crudo = (Apertura_decimal * 1000).astype(int)
                            
                            indice_actual, flex_actual, rot_calculada = vector_crudo[3], vector_crudo[4], vector_crudo[5]
                            delta_flex = flex_actual - flexion_anterior
                        
                            if indice_actual <= 10:
                                rotacion_retenida, rotacion_bloqueada = 1000, True
                            else:
                                if rotacion_retenida <= 150 or rotacion_bloqueada:
                                    if delta_flex < -15: rotacion_bloqueada = True
                                    elif delta_flex > 15: rotacion_bloqueada = False
                                else: rotacion_bloqueada = False

                            if abs(delta_flex) > 15: flexion_anterior = flex_actual
                            if not rotacion_bloqueada and abs(rot_calculada - rotacion_retenida) >= 50:
                                rotacion_retenida = rot_calculada

                            vector_crudo[5] = rotacion_retenida
                            VIS_VEC_CALCULADO = vector_crudo.copy()

            cv2.putText(color_image, "'e' Track | 'm' Modo | 'f' Gatillo | 's' Guardar | 'r' Rutinas | 'q' Salir", (10, 25), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.48, (0, 255, 0), 2)
            
            estado_txt = "EN VIVO" if VIS_TRACKING else "PAUSADO"
            color_estado = (0, 0, 255) if VIS_TRACKING else (0, 255, 0) 
            cv2.putText(color_image, f"Teleop: {estado_txt} | Modo {MODOS_TELECONTROL[VIS_MODO]}", (10, 50), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.52, color_estado, 2)

            if VIS_MODO == 1:
                est_pausa = ["P" if p else "L" for p in MOD_D_PAUSA]
                cv2.putText(color_image, f"Estado [P/L]: {est_pausa} | F_act: {MOD_F_REALES}", (10, 75),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 2)
            elif VIS_MODO == 2:
                txt_gat = "GATILLO ACTIVO (+F)" if VIS_GATILLO_FUERZA else "GATILLO SUELTO (BLOQ)"
                col_gat = (0, 255, 0) if VIS_GATILLO_FUERZA else (0, 0, 255)
                
                cv2.putText(color_image, f"Teclado 'f': {txt_gat}", (10, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.5, col_gat, 2)
                
                contacto_txt = [1 if b else 0 for b in MOD_O_DETECT]
                cv2.putText(color_image, f"F_Set: {MOD_F_SET} | Contacto: {contacto_txt}", (10, 100),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 0), 2)
            
            cv2.imshow('Vision 3D - Control de Mano', color_image)
            key = cv2.waitKey(1) & 0xFF
            
            if key == ord('f'):
                VIS_GATILLO_FUERZA = True
            else:
                VIS_GATILLO_FUERZA = False

            if key == ord('e'):
                VIS_TRACKING = not VIS_TRACKING
                print(f"\n>> MODO TELEOPERACIÓN: {'ACTIVADO' if VIS_TRACKING else 'PAUSADO'}")

            elif key == ord('m'):
                VIS_MODO = (VIS_MODO + 1) % 3
                VIS_REINICIAR = True 
                print(f"\n>> MODO CAMBIADO A: {MODOS_TELECONTROL[VIS_MODO]}")
            
            elif key == ord('s'):
                VIS_TRACKING = False 
                print("\n--- GUARDAR NUEVO GESTO ---")
                nombre = input("Ingrese nombre para este gesto: ").strip()
                if nombre:
                    existe = os.path.exists(ARCHIVO_GESTOS_3D)
                    with open(ARCHIVO_GESTOS_3D, mode='a', newline='') as file:
                        writer = csv.writer(file)
                        if not existe: writer.writerow(CABECERAS_GESTOS)
                        writer.writerow([nombre] + VIS_VEC_CALCULADO.tolist())
                    print(f" ✓ Gesto '{nombre}' guardado.")
                    
            elif key == ord('r'):
                VIS_TRACKING = False 
                print("\n" + "="*40 + "\n         MENÚ DE RUTINAS 3D\n" + "="*40)
                print("1. Ejecutar un gesto guardado (.csv)\n2. Crear y ejecutar NUEVA Rutina\n3. Importar Rutina\n4. Volver")
                opc = input("Seleccione: ")
                gestos = cargar_gestos()
                if opc == '1' and gestos:
                    for i, g in enumerate(gestos): print(f"{i}. {g[0]}")
                    try:
                        sel = int(input("Número de gesto (-1 cancelar): "))
                        if 0 <= sel < len(gestos): enviar_angulos(mano, [int(v) for v in gestos[sel][1:7]])
                    except ValueError: pass
                elif opc == '2' and gestos:
                    for i, g in enumerate(gestos): print(f"{i}. {g[0]}")
                    try:
                        indices = [int(x) for x in input("\nSecuencia (ej. 0,2,1): ").split(',') if x.strip()]
                        validos = [i for i in indices if 0 <= i < len(gestos)]
                        if validos:
                            if input("¿Guardar rutina? (s/n): ").lower() == 's':
                                exportar_rutina(input("Nombre: "), validos, gestos)
                            if input("¿Ejecutar ahora? (s/n): ").lower() == 's':
                                for idx in validos:
                                    enviar_angulos(mano, [int(v) for v in gestos[idx][1:7]])
                                    time.sleep(1.0)
                    except ValueError: pass
                elif opc == '3':
                    rutinas = cargar_rutinas()
                    if rutinas:
                        for i, r in enumerate(rutinas): print(f"{i}. {r[0]} | Secuencia: {r[2]}")
                        try:
                            sel = int(input("Número de rutina (-1 cancelar): "))
                            if 0 <= sel < len(rutinas):
                                for idx in [int(x) for x in rutinas[sel][1].split(',') if x.strip()]:
                                    enviar_angulos(mano, [int(v) for v in gestos[idx][1:7]])
                                    time.sleep(1.0)
                        except ValueError: pass
                print("\nRegresando a cámara...")
                
            elif key == ord('q'):
                VIS_SALIR = True
                break

    finally:
        pipeline.stop()
        cv2.destroyAllWindows()
        hilo_modbus.join()
        if mano is not None:
            try:
                mano._write_and_ack(REG_FORCE_SET, FUERZA_SEGURA)
                mano._write_and_ack(REG_SPEED_SET, VELOCIDADES_MAXIMAS)
            except Exception: pass
            enviar_angulos(mano, ABIERTO, timeout=2.0)
            mano.close()
        print("Cámara apagada y recursos liberados.")

## Ejecución del Programa

In [7]:
if __name__ == "__main__":
    main()

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1792785331.250961    9293 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1792785331.276564    9294 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


1. Usar calibración predeterminada
2. Calibrar mano (Recomendado)
Seleccione (1 o 2):  1



=== MODO ACTIVO | VELOCIDAD LIMITADA A 1000 ===
 'e' : Iniciar / Pausar Seguimiento
 'm' : Cambiar Modo (0: Estándar | 1: Pausa 0.5s | 2: Fuerza Incremental)
 'f' : MANTENER PRESIONADO para inyectar Fuerza en Modo 2 (Gatillo)
 's' : Guardar Gesto | 'r' : Menú Rutinas | 'q' : Salir

>> MODO TELEOPERACIÓN: ACTIVADO
Cámara apagada y recursos liberados.


## Sugerencias de Operación

1. **Retención Temporal del Gatillo `'f'` (Modo 2)**:
   - Debido a que `cv2.waitKey(1)` depende del retardo de repetición automática del sistema operativo ($\sim 250 - 500\,\text{ms}$ tras presionar una tecla), si se desea una rampa más continua sin micro-pausas al inicio de la pulsación, puede sustituirse el gatillo momentáneo por una ventana temporal (`tiempo_ultimo_f = time.time()`, manteniendo `VIS_GATILLO_FUERZA = (time.time() - tiempo_ultimo_f) < 0.25`) o por un interruptor (*toggle*).
2. **Memoria Histórica de Profundidad por Nodo**:
   - Actualmente, si uno de los 11 nodos arroja `z_metros == 0`, `prof_valida` pasa a `False` y se conserva íntegro el último vector válido `VIS_VEC_CALCULADO`. Almacenar un diccionario con la última profundidad válida por nodo (`ultimo_z[idx]`) permitiría actualizar la cinemática de los 10 nodos restantes sin descartar el fotograma completo.